## Test to analyze the data

We always start with a dataset to train on. Let's download the tiny shakespeare dataset
wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [1]:
# read it as an string object to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [3]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
print(chars)
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Basic tokenization of characters to integers

In [5]:
# create a mapping from characters to integers
stoi = {ch : i for i, ch in enumerate(chars)} # dictionary comprehension
itos = {i : ch for i, ch in enumerate(chars)}

# define encode and decode functions using lambda (e.g. shortcut for def encode(s): return [stoi[c] for c in s])
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # the 100 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first %90 will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    # picks a list of numbers of size = batch_size 
    # randomly chosen between 0, len(data) - block_size (to always recover full snippets)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

# same procedure above, match all possible context combinations to target
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [11]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


# Bigram Language Model Overview

This code implements a simple **Bigram Language Model** in PyTorch. A bigram model predicts the next token using only the current token:

$P(x_{t+1}|x_t)$


Unlike a general N-gram model, which uses the previous (N-1) tokens, a bigram model only considers one previous token.

---

## Model Parameters

The model contains a single learnable lookup table:

```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

This creates a matrix of shape

```text
(vocab_size, vocab_size)
```

where each row corresponds to one token and stores scores (logits) for all possible next tokens.

For example, with vocabulary

```text
0 → a
1 → b
2 → c
```

the table may look like:

```text
          next a   next b   next c
token a    0.2      1.5     -0.3
token b   -1.0      0.1      2.0
token c    1.7     -0.4      0.2
```

These values are initialized randomly and are learned during training.

---

## Training Example

Suppose the text is

```text
abca
```

Then

```python
xb = [[0,1,2]]     # input sequence "abc"
yb = [[1,2,0]]     # targets "bca"
```

meaning:

```text
a → b
b → c
c → a
```

---

## Forward Pass

Input:

```python
idx = [[0,1,2]]
```

has shape

```text
(B,T) = (1,3)
```

where

* B = batch size
* T = sequence length

The embedding layer replaces each token with its corresponding row:

```python
logits = self.token_embedding_table(idx)
```

producing

```text
[
 [
  [0.2, 1.5,-0.3],     # prediction after a
  [-1.0,0.1,2.0],      # prediction after b
  [1.7,-0.4,0.2]       # prediction after c
 ]
]
```

with shape

```text
(B,T,C) = (1,3,3)
```

where C is the vocabulary size.

---

## Loss Computation

Targets are

```python
targets = [[1,2,0]]
```

corresponding to

```text
after a → b
after b → c
after c → a
```

The tensors are flattened:

```python
logits  : (B*T,C)
targets : (B*T)
```

and cross entropy compares the predicted distributions against the correct next tokens.

During training, gradient descent updates the embedding table so that the correct next token receives higher scores.

---

## Text Generation

Suppose generation starts from

```python
idx = [[0]]
```

which corresponds to

```text
a
```

### Step 1: Compute predictions

```python
logits, _ = self(idx)
```

returns shape

```text
(B,T,C)
```

---

### Step 2: Keep only the last prediction

```python
logits = logits[:, -1, :]
```

Since only the newest token matters, this extracts the final time step and gives shape

```text
(B,C)
```

For example,

```text
[1.7,-0.4,0.2]
```

---

### Step 3: Convert scores to probabilities

```python
probs = F.softmax(logits, dim=-1)
```

which may produce

```text
P(a)=74%
P(b)=9%
P(c)=17%
```

---

### Step 4: Sample the next token

```python
idx_next = torch.multinomial(probs, num_samples=1)
```

Suppose token `a` is sampled:

```python
idx_next = [[0]]
```

with shape

```text
(B,1)
```

---

### Step 5: Append the new token

```python
idx = torch.cat((idx, idx_next), dim=1)
```

so

```text
a b c
```

becomes

```text
a b c a
```

with shape

```text
(B,T+1)
```

The process then repeats.

---

## Summary

The entire model performs:

```text
Current token
      ↓
Lookup corresponding row in embedding table
      ↓
Obtain scores for all possible next tokens
      ↓
Convert scores into probabilities
      ↓
Sample one token
      ↓
Append to sequence
      ↓
Repeat
```

This is the simplest possible language model. More advanced models such as transformers generalize this idea by using many previous tokens instead of only one.


In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


# How the Embedding Table is Updated During Training

For a Bigram Language Model, each token corresponds to one row of the embedding table. That row directly stores the logits (scores) for all possible next tokens.

For example, with vocabulary

```text
0 → a
1 → b
2 → c
```

the embedding table may be

```text
          next a   next b   next c
token a    0.2      1.5     -0.3
token b   -1.0      0.1      2.0
token c    1.7     -0.4      0.2
```

Each row represents the model's current belief about which token should follow the current token.

---

## Example: Training on a → b

Suppose we have one training example

```text
a → b
```

which corresponds to

```python
idx = [0]
target = [1]
```

The model looks up row 0:

```text
logits = [0.2, 1.5, -0.3]
```

These are scores for

```text
next a
next b
next c
```

---

## Softmax

Softmax converts the logits into probabilities:

```text
a : 0.19
b : 0.69
c : 0.12
```

Thus the model currently believes

```text
P(a|a)=19%
P(b|a)=69%
P(c|a)=12%
```

The correct answer is

```text
b
```

whose one-hot representation is

```text
[0,1,0]
```

---

## Cross Entropy Gradient

For softmax followed by cross entropy,

```text
gradient = probabilities - target
```

Therefore

```text
[0.19, 0.69, 0.12]
-
[0,    1,    0]
=
[0.19, -0.31, 0.12]
```

Interpretation:

* Increase the score for the correct token (b).
* Decrease the scores for the incorrect tokens (a and c).

---

## Gradient Descent Update

Suppose the learning rate is

```text
lr = 0.1
```

Gradient descent performs

```text
new_weight = old_weight - lr × gradient
```

Starting from

```text
[0.2, 1.5, -0.3]
```

and using gradient

```text
[0.19, -0.31, 0.12]
```

gives

```text
[0.181, 1.531, -0.312]
```

The score for the correct next token increases, while the scores for incorrect tokens decrease.

---

## Why the Entire Row is Updated

The row itself is the logits vector:

```text
token a
↓
[score for next a,
 score for next b,
 score for next c]
```

Since every component affects the softmax probabilities and the loss, every component receives a gradient and is updated.

---

## Batch Training

Suppose

```python
B = 2
T = 3
```

with

```python
xb =
[
 [0,1,2],
 [1,2,0]
]

yb =
[
 [1,2,0],
 [2,0,1]
]
```

This represents six independent training examples:

```text
a→b
b→c
c→a
b→c
c→a
a→b
```

Thus

```text
B × T = 6
```

parallel next-token prediction problems are solved simultaneously.

The tensor

```text
(B,T,C)
```

is flattened into

```text
(B×T,C)
```

and cross entropy computes the average loss over all B×T examples.

---

## Gradient Accumulation

If a row appears multiple times in a batch, gradients from all occurrences are summed together.

For example,

```text
a→b
a→b
a→c
```

all contribute to the gradient of row "a":

```text
grad(a)
=
grad₁
+
grad₂
+
grad₃
```

After all gradients are accumulated,

```python
optimizer.step()
```

updates the embedding matrix.

---

## Summary

For a Bigram Language Model, training consists of

```text
current token
      ↓
select corresponding row of embedding matrix
      ↓
obtain logits for all possible next tokens
      ↓
softmax
      ↓
cross entropy loss
      ↓
gradient = probabilities − target
      ↓
update the entire row
```

Thus, a Bigram Language Model learns a transition probability matrix by repeatedly adjusting each row so that the correct next token receives higher probability.

In [13]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

2.382369041442871

lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


# Self Attention Trick: Accumulate Information From Other Tokens

## Basic example: matrix multiplication as weighted average

For the given forms of a and b, the matrix products a @ b will result in the following 2x3 matrix:

Each row will be the average of all prior rows and the row itself averaged from the original b matrix, more specifically 

For a given sequence in a batch, the representation at position t becomes the average of the current token vector and all previous token vectors.

In [14]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [15]:
# consider the following toy example:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels (token representation vector size)
x = torch.randn(B,T,C)
print(x.shape)

# version 1: we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
print(torch.allclose(xbow, xbow2))

# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3))

torch.Size([4, 8, 2])
True
True


# Self-Attention: Learned Information Routing Between Tokens

Self-attention is the core mechanism that lets tokens communicate with each other inside a transformer.

In the previous weighted-average example, each token simply averaged all previous token vectors equally:

$x_{\text{out},t} = \frac{1}{t+1}\sum_{i=0}^{t}x_i$

This was a fixed rule. Token (t) received information from all previous tokens with equal weight.

Self-attention replaces this fixed averaging rule with a learned, input-dependent communication system. Instead of saying:

```text
Average all previous tokens equally.
```

it learns:

```text
Which previous tokens are relevant?
How strongly are they relevant?
What information should be transferred from them?
```

---

## Abstract Mathematical Definition

Let the input be:

$X \in \mathbb{R}^{B \times T \times C}$

where:

```text
B = batch size
T = sequence length
C = token embedding dimension
```

Each token vector is:

$x_i \in \mathbb{R}^{C}$

For one attention head, we choose a head size:

$H = \text{head size}$

The model learns three linear projections:

$W_Q \in \mathbb{R}^{C \times H}$

$W_K \in \mathbb{R}^{C \times H}$

$W_V \in \mathbb{R}^{C \times H}$

These produce:

$Q = XW_Q$

$K = XW_K$

$V = XW_V$

with:

$Q,K,V \in \mathbb{R}^{B \times T \times H}$

For each token:

$q_i = x_iW_Q$

$k_i = x_iW_K$

$v_i = x_iW_V$

The attention score between token (i) and token (j) is:

$S_{ij} = q_i \cdot k_j$

This asks:

```text
How relevant is token j to token i?
```

Then the full attention score matrix is:

$S = QK^T$

In standard scaled attention:

$S = \frac{QK^T}{\sqrt{H}}$

The division by $\sqrt{H}$ keeps dot products numerically stable when the head size is large.

After causal masking and softmax:

$A = \operatorname{softmax}(\operatorname{mask}(S))$

Then the output is:

$O = AV$

Equivalently, for each token:

$o_i = \sum_{j \leq i} A_{ij}v_j$

So the new representation of token (i) is a weighted mixture of the value vectors of relevant previous tokens.

---

## Core Concept

Self-attention creates a learned communication graph between tokens.

```text
Tokens are nodes.
Attention weights are edges.
Value vectors are the information flowing through the edges.
```

The matrix:

$A = \operatorname{softmax}(QK^T)$

decides who communicates with whom.

The product:

$O = AV$

moves information through that communication graph.

So one attention head performs:

```text
token embeddings
      ↓
query/key/value projections
      ↓
query-key similarity scores
      ↓
dynamic attention weights
      ↓
weighted transfer of value information
      ↓
new contextual token representations
```

---

## Query, Key, and Value as Information Subspaces

Each token originally lives in the full embedding space:

$x_i \in \mathbb{R}^{C}$

Self-attention projects the same token into three different learned subspaces:

$q_i = x_iW_Q$

$k_i = x_iW_K$

$v_i = x_iW_V$

These three projections have different roles.

```text
Query:
What information is this token looking for?

Key:
What type of information does this token advertise?

Value:
What information does this token actually communicate?
```

So (Q) and (K) do not directly carry the final transmitted information. They construct the relevance structure.

The values (V) carry the actual information that gets passed between tokens.

Street version:

```text
Q and K decide the pipes.
V carries the water through the pipes.
```

More precise version:

```text
Q and K determine who communicates with whom.
V determines what gets communicated.
```

---

## Hidden Information Space Inside the Dot Product

The score:

$S_{ij} = q_i \cdot k_j$

is a single number, but it is produced by comparing two vectors in an (H)-dimensional learned space:

$q_i,k_j \in \mathbb{R}^{H}$

Expanded:

$S_{ij} = \sum_{\alpha=1}^{H}q_{i,\alpha}k_{j,\alpha}$

So the dot product hides an internal (H)-dimensional comparison space.

The final output of the dot product is one relevance score, but that score is produced by comparing many learned features.

Street interpretation:

```text
The attention head creates a private H-dimensional language for deciding whether two tokens are relevant.
The dot product compresses that hidden comparison into one compatibility score.
```

This is why head size matters.

---

## Head Size vs Embedding Dimension

In the code:

```python
B,T,C = 4,8,32
head_size = 16
```

Each token starts as a 32-dimensional vector:

$x_i \in \mathbb{R}^{32}$

but each attention head projects it into a 16-dimensional query/key/value space:

$q_i,k_i,v_i \in \mathbb{R}^{16}$

Therefore:

```text
C = full token representation size
head_size = dimension of one attention head's communication subspace
```

The sequence length (T) counts how many tokens exist.

The head size (H) counts how many internal features one head uses to compare and communicate between tokens.

So:

```text
T = number of tokens
C = full embedding size per token
H = size of one attention head's hidden communication space
```

The head size does not need to equal (C). One head usually uses only part of the full representation space.

---

## Why Head Size Can Be Smaller Than C

The full token vector may contain many types of information:

```text
word identity
position
syntax
semantics
context
```

One attention head does not need to use the entire representation. It extracts a smaller learned communication channel from the full token vector.

So one head performs:

$\mathbb{R}^{C} \to \mathbb{R}^{H}$

In the code:

$\mathbb{R}^{32} \to \mathbb{R}^{16}$

This creates a specialized subspace for one kind of token-token communication.

In multi-head attention, this becomes more important.

For example, if:

```text
C = 32
num_heads = 4
head_size = 8
```

then each head produces an 8-dimensional output, and concatenating all heads gives:

$4 \times 8 = 32$

So multiple smaller heads create multiple communication subspaces whose outputs combine back into the full embedding dimension.

---

## Information Space Expansion

Even when (H < C), self-attention is not simply throwing information away.

It creates multiple structured views of the same token:

```text
query view
key view
value view
```

The same token now participates in three roles:

```text
as a receiver of information
as an advertiser of relevance
as a sender of content
```

This creates a richer internal structure than a plain token embedding.

The transformer is not merely storing words as vectors. It is constructing learned information channels where tokens dynamically route information to each other.

Street interpretation:

```text
A transformer builds learned pipes and channels inside a deeper information space.
Q and K decide where the pipes connect.
V carries the actual information through those pipes.
Each layer updates the token embeddings by mixing information from relevant tokens.
```

---

## Toy Example

Suppose the sequence is:

```text
car gas low
```

The fixed weighted-average model would update `low` as:

$o_{\text{low}} = \frac{1}{3}x_{\text{car}} + \frac{1}{3}x_{\text{gas}} + \frac{1}{3}x_{\text{low}}$

This treats all previous tokens equally.

Self-attention can instead learn:

```text
car: 0.25
gas: 0.65
low: 0.10
```

Then:

$o_{\text{low}} = 0.25v_{\text{car}} + 0.65v_{\text{gas}} + 0.10v_{\text{low}}$

This means that when updating the representation of `low`, the model receives mostly information from `gas`, some information from `car`, and a little information from itself.

The query-key mechanism decides that `gas` is highly relevant to `low`.

The value vector of `gas` carries useful learned information such as:

```text
fuel
engine
empty
running low
car-related context
```

So the token `low` becomes contextually enriched by information from `gas` and `car`.

---

## Relation to the Code

The code implements one self-attention head:

$Q = XW_Q$

$K = XW_K$

$V = XW_V$

$A = \operatorname{softmax}(\operatorname{mask}(QK^T))$

$O = AV$

The important conceptual change from the earlier weighted-average code is that the weight matrix is no longer fixed.

Earlier:

```text
weights were manually defined averages
```

Now:

```text
weights are dynamically produced from token-token relevance
```

Thus self-attention is a learned weighted aggregation mechanism.

---

## Final Summary

Self-attention is a learned communication mechanism.

```text
Q-space:
what each token is looking for

K-space:
what each token advertises

QKᵀ:
dynamic relevance matrix

softmax:
turn relevance scores into communication weights

V-space:
what information each token sends

AV:
weighted information transfer

O:
new contextual token representations
```

At the highest level:

```text
Self-attention constructs learned information subspaces and dynamic communication paths between tokens.
Q and K build the relevance structure.
V carries the information.
The output is a richer contextual embedding of each token.
```

This is the core mechanism behind transformer architecture.



In [16]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

# Key Notes on Attention in Transformers

Attention is a learned communication mechanism between tokens. Each token can look at other tokens, decide how relevant they are, and aggregate information from them using a weighted sum.

Mathematically, a single attention head computes:

$Q = XW_Q$

$K = XW_K$

$V = XW_V$

$A = \operatorname{softmax}\left(\operatorname{mask}\left(\frac{QK^T}{\sqrt{H}}\right)\right)$

$O = AV$

where:

```text
Q = queries: what each token is looking for
K = keys: what each token advertises
V = values: what each token communicates
A = attention weights: who listens to whom
O = output: updated contextual token representations
```

---

## Attention as Communication

Attention can be viewed as a directed graph.

```text
tokens = nodes
attention weights = directed edges
value vectors = information flowing through edges
```

If token (i) attends strongly to token (j), then token (i) receives a lot of information from token (j).

For token (i), the output is:

$o_i = \sum_j A_{ij}v_j$

This means:

```text
new token i representation
=
weighted sum of information from other tokens
```

Street interpretation:

```text
Q and K decide the pipes.
V carries the water through the pipes.
```

More precise interpretation:

```text
Q and K determine who communicates with whom.
V determines what information is communicated.
```

---

## No Built-In Notion of Space or Order

Attention itself does not naturally know token order. It simply operates on a set of vectors.

For example, without positional information, the tokens in:

```text
dog bites man
```

and

```text
man bites dog
```

are just the same token vectors arranged differently. Attention can compare vectors, but it does not inherently know which token is first, second, or third.

That is why transformers add positional embeddings:

$x = \text{token embedding} + \text{position embedding}$

The token embedding tells the model what the token is.

The position embedding tells the model where the token is.

So the transformer input contains both:

```text
token identity + token location
```

This allows the model to learn order-sensitive structure, such as:

```text
subject before verb
object after verb
earlier context before later token
opening bracket before closing bracket
```

---

## Basic Example: Why Position Matters

Consider:

```text
dog bites man
```

The meaning is different from:

```text
man bites dog
```

The words are the same, but the order changes the meaning.

Without positional embeddings, attention mainly sees the set:

```text
{dog, bites, man}
```

With positional embeddings, it can distinguish:

```text
dog at position 0
man at position 2
```

from:

```text
man at position 0
dog at position 2
```

So positional embeddings give attention the ability to learn grammar and sequence structure.

---

## Batch Dimension: Examples Do Not Communicate

If the input has shape:

$X \in \mathbb{R}^{B \times T \times C}$

then:

```text
B = batch size
T = sequence length
C = embedding dimension
```

Each batch example is processed independently.

For example, if:

```text
B = 4
```

then there are 4 separate sequences being processed in parallel.

Attention produces:

$A \in \mathbb{R}^{B \times T \times T}$

This means each batch example gets its own attention matrix.

```text
batch 0 has its own T x T attention matrix
batch 1 has its own T x T attention matrix
batch 2 has its own T x T attention matrix
batch 3 has its own T x T attention matrix
```

Tokens from different batch examples never talk to each other. The batch dimension is only for parallel computation.

---

## Encoder Attention vs Decoder Attention

The difference between encoder-style attention and decoder-style attention is masking.

### Encoder Attention

In an encoder block, tokens are usually allowed to communicate with all other tokens.

There is no triangular causal mask.

For sequence length 4, full attention allows:

```text
[1 1 1 1]
[1 1 1 1]
[1 1 1 1]
[1 1 1 1]
```

Every token can see every other token.

This is useful when the full input is already available, such as in BERT-style models or the encoder of a translation model.

---

### Decoder Attention

In a decoder block, future tokens are hidden using a triangular mask.

The mask looks like:

```text
[1 0 0 0]
[1 1 0 0]
[1 1 1 0]
[1 1 1 1]
```

This means:

```text
token 0 can only see token 0
token 1 can see token 0 and token 1
token 2 can see token 0, token 1, and token 2
token 3 can see token 0, token 1, token 2, and token 3
```

This is necessary for autoregressive language modeling, where the model predicts the next token without cheating by looking into the future.

GPT-style models use decoder-style masked self-attention.

---

## Self-Attention vs Cross-Attention

### Self-Attention

In self-attention, queries, keys, and values all come from the same source (X):

$Q = XW_Q$

$K = XW_K$

$V = XW_V$

So the sequence attends to itself.

Conceptually:

```text
Which parts of myself should I listen to?
```

---

### Cross-Attention

In cross-attention, queries come from one source, but keys and values come from another source.

For example, in an encoder-decoder transformer:

$Q = X_{\text{decoder}}W_Q$

$K = X_{\text{encoder}}W_K$

$V = X_{\text{encoder}}W_V$

The decoder asks questions using queries, while the encoder provides keys and values.

Conceptually:

```text
Decoder query:
What source information do I need right now?

Encoder keys and values:
Here is the source sentence information.
```

So:

```text
self-attention = one sequence attends to itself
cross-attention = one sequence attends to another sequence
```

---

## Scaled Attention

Raw attention scores are computed as:

$S = QK^T$

For one token pair:

$S_{ij} = q_i \cdot k_j$

If:

$q_i,k_j \in \mathbb{R}^{H}$

then:

$S_{ij} = \sum_{\alpha=1}^{H} q_{i,\alpha}k_{j,\alpha}$

As the head size (H) grows, this dot product can become large in magnitude. Large attention scores make softmax too sharp.

For example:

```text
softmax([0.2, 0.5, 0.1])
```

is smooth and distributed, but:

```text
softmax([2, 8, -3])
```

is almost one-hot.

If softmax becomes too sharp too early, one token dominates the attention distribution and gradients become less useful.

To prevent this, scaled attention uses:

$S = \frac{QK^T}{\sqrt{H}}$

In code:

```python
wei = q @ k.transpose(-2, -1) * head_size**-0.5
```

This keeps attention scores numerically stable and prevents softmax from saturating too much.

---

## Full Architecture Walkthrough

For a language model, the flow is:

```text
token IDs
   ↓
token embeddings
   ↓
add positional embeddings
   ↓
self-attention
   ↓
contextual token representations
   ↓
feed-forward layers
   ↓
logits for next-token prediction
```

More concretely:

```python
tok_emb = token_embedding_table(idx)
pos_emb = position_embedding_table(torch.arange(T))
x = tok_emb + pos_emb
```

Now each token vector contains:

```text
what the token is
+
where the token is
```

Then self-attention computes:

```python
q = query(x)
k = key(x)
v = value(x)

wei = q @ k.transpose(-2, -1) * head_size**-0.5
wei = masked_softmax(wei)
out = wei @ v
```

The output `out` contains updated token representations. Each token has gathered information from relevant previous tokens.

---

## Final Summary

Attention is a learned information-routing mechanism.

```text
token embeddings:
represent what each token is

positional embeddings:
represent where each token is

Q and K:
construct relevance scores between tokens

masking:
controls which tokens are allowed to communicate

softmax:
turns relevance scores into communication weights

V:
contains the information being communicated

AV:
aggregates information from relevant tokens
```

The transformer repeatedly applies this process so that token embeddings become increasingly contextual.

At a high level:

```text
Attention builds dynamic communication paths between tokens.
Position embeddings give the paths order.
Masking controls which paths are allowed.
Values carry the actual information.
The output is a richer contextual representation of each token.
```

In [17]:
# visualization of scaling
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

print(k.var())
print(q.var())
print(wei.var())

print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1))

print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)) # gets too peaky, converges to one-hot

tensor(1.0449)
tensor(1.0700)
tensor(1.0918)
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


# Multi-Head Self-Attention: Multiple Learned Communication Subspaces

A single attention head creates one learned communication channel between tokens. It computes one attention matrix, meaning it has one pattern for deciding which tokens should communicate with which other tokens.

Multi-head attention extends this by using several attention heads in parallel. Each head has its own query, key, and value projections, so each head can learn a different relevance structure and a different information subspace.

High-level idea:

```text
Single head:
one communication pattern

Multi-head attention:
many communication patterns in parallel
```

Each head can specialize in a different kind of token-token relationship. For example, one head may learn semantic relations, another may learn syntax-like relations, another may learn positional patterns, and another may learn long-range dependencies. These roles are not manually assigned; they emerge through training.

---

## Why Not Use One Massive Head?

A single massive head can have a large value space, but it still produces only one attention matrix.

For one head:

```text
A = softmax(QK^T)
O = AV
```

There is only one attention matrix $A$.

That means the same token-to-token attention weights are used for all value dimensions inside that head.

So if token `low` attends to:

```text
car: 0.3
gas: 0.6
low: 0.1
```

then this same attention pattern is used across that head's whole value vector.

Multi-head attention is different. It creates multiple attention matrices:

```text
Head 1: A_1
Head 2: A_2
Head 3: A_3
...
```

Each head can route information differently.

For example, in the phrase:

```text
car gas low
```

one head may learn:

```text
Head 1:
low attends mostly to gas
```

while another head may learn:

```text
Head 2:
low attends mostly to car
```

So multi-head attention is better understood as many smaller learned communication channels rather than one huge channel.

Street version:

```text
One massive head = one huge pipe with one routing decision.

Multi-head attention = many smaller pipes, each with its own routing decision.
```

---

## Shape Example

Suppose:

```python
B = 1
T = 3
n_embd = 4
n_head = 2
head_size = 2
```

The input has shape:

```text
x: (B, T, n_embd) = (1, 3, 4)
```

Each head receives the same input `x`, but has its own learned projections.

Each head outputs:

```text
head 1 output: (1, 3, 2)
head 2 output: (1, 3, 2)
```

Then we concatenate along the last dimension, the embedding/channel dimension:

```python
out = torch.cat([head1_out, head2_out], dim=-1)
```

Result:

```text
out: (1, 3, 4)
```

So concatenation is not along batch dimension and not along time dimension.

It is along feature dimension:

```text
(B, T, head_size) + (B, T, head_size)
→
(B, T, n_head * head_size)
```

For one token:

```text
head1 output = [0.8, 0.6]
head2 output = [0.9, 0.2]
```

After concatenation:

```text
[0.8, 0.6, 0.9, 0.2]
```

This combines the outputs of different heads side by side.

---

## Projection After Concatenation

After concatenating the heads, we usually apply a linear projection:

```python
self.proj = nn.Linear(n_embd, n_embd)
```

This projection is not another attention operation. It does not compute a weighted average over tokens.

Attention mixes information across tokens.

Projection mixes information across feature channels and heads.

After concatenation, the output looks like:

```text
[head 1 information | head 2 information | head 3 information | ...]
```

The projection learns how to recombine those separate head outputs into one unified token representation.

For one token:

```text
concat = [a1, a2, b1, b2]
```

The projection learns:

```text
new_feature_1 = learned mix of a1, a2, b1, b2
new_feature_2 = learned mix of a1, a2, b1, b2
new_feature_3 = learned mix of a1, a2, b1, b2
new_feature_4 = learned mix of a1, a2, b1, b2
```

So the projection creates a learned recombination space for the contextual information produced by different heads.

Street version:

```text
Heads collect different messages.
Concatenation stacks the messages side by side.
Projection mixes them into one updated token vector.
```

Important distinction:

```text
Attention heads mix information across tokens.
Projection mixes information across heads/features.
```

---

## Why Output Back to n_embd?

The transformer block uses residual connections:

```python
x = x + self_attention(x)
```

For this addition to work, both tensors must have the same shape.

So if the input is:

```text
x: (B, T, n_embd)
```

then multi-head attention must also output:

```text
(B, T, n_embd)
```

That is why we usually choose:

```text
head_size = n_embd // n_head
```

so that:

```text
n_head * head_size = n_embd
```

Example:

```text
n_embd = 32
n_head = 4
head_size = 8

4 heads × 8 dimensions = 32 dimensions
```

The model keeps the same embedding dimension while creating multiple internal communication subspaces.

---

## Final Conceptual Summary

Multi-head attention creates several learned contextual subspaces in parallel.

Each head has its own:

```text
query space
key space
value space
attention matrix
communication pattern
```

Then all head outputs are concatenated along the embedding dimension.

Finally, a learned projection mixes the head outputs back into one unified representation.

The full idea is:

```text
Input token representations
        ↓
multiple attention heads in parallel
        ↓
each head builds its own contextual subspace
        ↓
concatenate head outputs along feature dimension
        ↓
projection recombines head information
        ↓
updated token representation
```

So multi-head attention is not just “more dimensions.” It is multiple independently learned ways of routing information between tokens.


```python
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        # self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        # out = self.dropout(self.proj(out))
        return out
```

# Feedforward Layer in a Transformer Block

After self-attention, each token has already gathered contextual information from other relevant tokens.

For example, in the phrase:

```text
car gas low
```

the token `low` may have already received information from `car` and `gas` through attention.

So after attention, the representation of `low` is no longer just the isolated token `low`. It may contain contextual information like:

```text
low + gas context + car context
```

The feedforward layer then processes this updated token vector.

High-level idea:

```text
Attention:
tokens communicate with each other

Feedforward:
each token individually processes what it received
```

---

## What the Feedforward Layer Does

A typical transformer feedforward layer is:

```python
nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd),
)
```

or sometimes GELU is used instead of ReLU.

If each token vector has size:

```text
n_embd = 384
```

then the feedforward layer maps:

```text
384 → 1536 → 384
```

So the token is first expanded into a larger temporary space, transformed nonlinearly, and then projected back to the original embedding size.

---

## Learnable Space Interpretation

The feedforward layer creates a larger learnable transformation space for each token.

The first linear layer expands the token vector:

```text
original contextual token space
        ↓
larger hidden feature space
```

The nonlinearity allows the model to build richer features.

The second linear layer compresses the result back:

```text
larger learned feature space
        ↓
original embedding dimension
```

So the feedforward layer is like a local reasoning/computation module for each token.

Street version:

```text
Attention lets the token listen.
Feedforward lets the token think about what it heard.
```

---

## Important: It Does Not Mix Tokens

If the input shape is:

```text
x: (B, T, n_embd)
```

then the feedforward layer maps:

```text
(B, T, n_embd)
→
(B, T, 4*n_embd)
→
(B, T, n_embd)
```

It does not mix across the time dimension `T`.

That means token 5 does not directly look at token 3 inside the feedforward layer.

Instead:

```text
attention already mixed information across tokens
feedforward transforms each token independently afterward
```

So the same MLP is applied separately to every token position.

---

## Basic Example

Suppose after attention, the token `low` has a contextual vector containing information from `car` and `gas`:

```text
low_context = [low meaning, gas information, car information, position information]
```

The feedforward layer may transform this into a richer feature like:

```text
"running low on fuel"
```

Conceptually:

```text
low + gas + car
      ↓
feedforward MLP
      ↓
low-fuel contextual feature
```

The model learns this transformation from training data.

---

## Why Expand to 4 * n_embd?

The expansion gives the model more temporary workspace.

If the model only did:

```text
n_embd → n_embd
```

then the transformation would be more limited.

Using:

```text
n_embd → 4*n_embd → n_embd
```

lets the model create many intermediate features before compressing back.

For example, the hidden layer may represent possible features such as:

```text
fuel-related context
syntax role
object relation
phrase meaning
next-token clues
```

These are not manually programmed. The model learns useful internal features during training.

---

## Relation to the Transformer Block

A transformer block alternates between communication and computation:

```text
self-attention
      ↓
tokens exchange information

feedforward
      ↓
each token processes its updated information
```

With residual connections, the block usually does:

```python
x = x + self_attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

So the feedforward layer does not replace the token representation completely. It adds a learned update to it.

---

## Final Summary

The feedforward layer is a position-wise MLP applied independently to each token.

It creates a larger learnable nonlinear space where each token can transform its contextual information.

```text
Attention:
builds context by communicating across tokens

Feedforward:
transforms each token's contextual representation locally

Expansion:
creates temporary extra feature space

Projection back:
returns to original embedding dimension so the transformer block shape stays consistent
```

In short:

```text
Multi-head attention gathers information.
Feedforward layers process and refine that information.
```


```python
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            # nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
```

# Residual Connections: Why They Make Deep Networks Trainable

Residual connections are skip connections that add the input of a layer back to its output.

Instead of replacing the representation completely:

```python
x = F(x)
```

we do:

```python
x = x + F(x)
```

In a transformer block, this usually appears as:

```python
x = x + self_attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

The basic information-preservation idea is simple: the model keeps the old representation and adds a learned update on top of it.

But the deeper reason residual connections matter is that they help gradients flow backward through deep networks.

---

## The Problem Without Residual Connections

Without residual connections, a deep network is a long chain of transformations:

```text
x0 → F1 → F2 → F3 → ... → loss
```

During backpropagation, the gradient must pass through every layer.

Mathematically, the gradient contains a product of many derivatives:

$\frac{\partial L}{\partial x_0} = \frac{\partial L}{\partial x_N} J_N J_{N-1} \cdots J_1$

where $J_i$ is the derivative/Jacobian of layer $F_i$.

This can become unstable.

If many derivatives are smaller than 1, the gradient shrinks:

```text
0.5 × 0.5 × 0.5 × 0.5 × ... → almost 0
```

This is called the vanishing gradient problem.

If many derivatives are larger than 1, the gradient can explode.

So in very deep networks, early layers may receive weak or unstable learning signals.

---

## How Residual Connections Help

A residual layer has the form:

$y = x + F(x)$

Now the derivative is:

$\frac{dy}{dx} = I + J_F$

where:

```text
I = identity path
J_F = derivative of the learned transformation
```

The important part is the identity term $I$.

This means the gradient has a direct path backward through the network, even if $F(x)$ is badly initialized, noisy, or hard to optimize.

Street version:

```text
Without residual:
the gradient must survive every layer.

With residual:
the gradient has a shortcut around every layer.
```

So residual connections act like gradient highways.

---

## Simple Scalar Example

Without residual connection:

$y = 0.5x$

Derivative:

$\frac{dy}{dx} = 0.5$

After 10 layers:

$0.5^{10} = 0.000976$

The gradient becomes almost zero.

---

With residual connection:

$y = x + 0.01x$

So:

$y = 1.01x$

Derivative:

$\frac{dy}{dx} = 1.01$

The gradient stays close to 1 instead of collapsing.

This makes optimization much easier.

---

## Why This Matters in Transformers

A transformer block usually does:

```python
x = x + attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

So each block says:

```text
keep the current representation
add communication update
keep the current representation
add computation update
```

The attention layer adds information gathered from other tokens.

The feedforward layer adds a local transformation of each token.

But neither one has to fully reconstruct the entire representation from scratch.

Each layer only needs to learn a useful correction.

---

## Deep Network Interpretation

Without residuals, a deep network is like:

```text
representation → replace → replace → replace → replace
```

Every layer must preserve useful information while also transforming it.

With residuals, the network becomes more like:

```text
representation
+ update 1
+ update 2
+ update 3
+ update 4
...
```

Not exactly this simple mathematically, but conceptually this is the right picture.

Each layer refines the representation instead of completely rewriting it.

---

## Final Summary

Residual connections make deep networks trainable because they:

```text
preserve forward information
give gradients a direct backward path
reduce vanishing-gradient problems
make each layer learn an update instead of a full replacement
allow very deep stacks of transformer blocks to train stably
```

Street version:

```text
Residual connections let the model keep what it already knows and add new information gradually.
They also give gradients a highway backward, so early layers can still learn even in a deep network.
```

In short:

```text
residual connection = old representation + learned update
```

This is one of the main reasons deep transformer architectures can be trained successfully.

---

## Residual Connections and Their Connection to Time-Dependent Perturbation Theory

Residual connections primarily address the **vanishing gradient problem** by introducing an identity path:

$$
x_{l+1} = x_l + F(x_l).
$$

This allows gradients to propagate through the network via

$$
\frac{\partial L}{\partial x_l}
=
\frac{\partial L}{\partial x_{l+1}}
\left(
I + \frac{\partial F}{\partial x_l}
\right),
$$

so even if the Jacobian of $F$ becomes small, the identity term $I$ preserves gradient flow.

### Analogy with Time-Dependent Perturbation Theory

The analogy is closer to **time-dependent perturbation theory** than to stationary perturbation theory.

In quantum mechanics, the state evolves according to

$$
i\hbar \frac{d}{dt}|\psi(t)\rangle
=
H(t)|\psi(t)\rangle,
$$

whose infinitesimal evolution is

$$
|\psi(t+\Delta t)\rangle
=
\left(
I
-
\frac{i}{\hbar}H(t)\Delta t
\right)
|\psi(t)\rangle.
$$

Similarly, a residual block may be written as

$$
x_{l+1}
=
\left(
I+\Delta t\,A_l
\right)x_l,
$$

where the layer index $l$ plays the role of time and $A_l$ is analogous to a time-dependent Hamiltonian.

Stacking many residual blocks gives

$$
x_N
=
\prod_l
\left(
I+\Delta t\,A_l
\right)x_0.
$$

In the continuous-depth limit,

$$
x(T)
=
\mathcal T
\exp
\left(
\int_0^T A(t)\,dt
\right)x_0,
$$

which is mathematically analogous to the quantum evolution operator

$$
U(T,0)
=
\mathcal T
\exp
\left(
-\frac{i}{\hbar}
\int_0^T H(t)\,dt
\right).
$$

Expanding the time-ordered exponential yields the Dyson series,

$$
U(T,0)
=
I
+
\left(
-\frac{i}{\hbar}
\right)
\int H(t_1)\,dt_1
+
\left(
-\frac{i}{\hbar}
\right)^2
\int\!\!\int
H(t_1)H(t_2)\,dt_1dt_2
+\cdots,
$$

showing that deep residual networks can be interpreted as accumulating successive small perturbative corrections around the identity.

### Correspondence

| Quantum Mechanics | Residual Networks |
|------------------|------------------|
| Time $t$ | Depth $l$ |
| State $\|\psi(t)\rangle$ | Feature vector $x_l$ |
| Hamiltonian $H(t)$ | Layer operator $A_l$ |
| Infinitesimal propagator $I-\frac{i}{\hbar}H(t)\Delta t$ | Residual block $I+\Delta tA_l$ |
| Time-ordered exponential | Composition of residual blocks |
| Dyson series | Deep residual expansion |

### Exploding Gradients

Residual connections mainly help with **vanishing gradients**. They do not fundamentally prevent exploding gradients, since repeated multiplication by large Jacobians can still amplify gradients.

In practice, exploding gradients are controlled by techniques such as

- Batch Normalization,
- Layer Normalization,
- Gradient clipping,
- Proper weight initialization,
- Adaptive optimizers.

These stabilization mechanisms are separate from the perturbative interpretation above and should not be regarded as part of the analogy with time-dependent perturbation theory.

# LayerNorm vs BatchNorm in Transformers

Normalization helps deep neural networks train more stably. In transformers, the most important normalization method is **LayerNorm**.

For transformer activations, the tensor usually has shape:

```text
x.shape = (B, T, C)
```

where:

```text
B = batch size
T = sequence length / number of tokens
C = embedding dimension / feature dimension
```

So:

```text
x[b, t]
```

is one token vector of size `C`.

---

## What LayerNorm Does

LayerNorm normalizes each token vector independently across the feature dimension `C`.

So for one token vector:

```text
x[b, t] = [2.0, -1.0, 5.0, 3.0]
```

LayerNorm computes the mean and variance of this vector itself, then normalizes it.

Conceptually:

```text
For every token independently:
normalize its feature vector.
```

So LayerNorm does not mix information across:

```text
batch examples
time positions
other tokens
```

It only normalizes inside each token vector.

---

## Small LayerNorm Example

Suppose one token vector is:

```text
x = [2, -1, 5]
```

Mean:

```text
mean = (2 + (-1) + 5) / 3 = 2
```

Subtract the mean:

```text
[2, -1, 5] - 2 = [0, -3, 3]
```

Variance:

```text
var = (0² + (-3)² + 3²) / 3 = 6
```

Standard deviation:

```text
std = sqrt(6) ≈ 2.45
```

Normalize:

```text
[0, -3, 3] / 2.45 ≈ [0, -1.22, 1.22]
```

Then LayerNorm applies learned scale and shift parameters:

```text
output = gamma * normalized_x + beta
```

At initialization, usually:

```text
gamma = 1
beta = 0
```

but training learns better values.

---

## What BatchNorm Does

BatchNorm normalizes using statistics from the batch.

For a simple tensor:

```text
x.shape = (B, C)
```

BatchNorm normalizes each feature/channel using values from many examples in the batch.

Example:

```text
example 1 = [2, -1, 5]
example 2 = [4,  1, 7]
```

BatchNorm looks across the batch for each feature.

For channel 0:

```text
values = [2, 4]
mean = 3
```

For channel 1:

```text
values = [-1, 1]
mean = 0
```

For channel 2:

```text
values = [5, 7]
mean = 6
```

So BatchNorm says:

```text
Normalize each feature using statistics from the batch.
```

LayerNorm instead says:

```text
Normalize each token vector by itself.
```

---

## Main Difference

For transformer-shaped data:

```text
x.shape = (B, T, C)
```

LayerNorm normalizes across:

```text
C dimension
```

for each token independently.

BatchNorm normalizes across:

```text
B dimension, and sometimes T depending on implementation
```

for each feature/channel.

Clean comparison:

```text
LayerNorm:
"Normalize this token's feature vector."

BatchNorm:
"Normalize this feature using other examples in the batch."
```

---

## Why Transformers Prefer LayerNorm

Transformers usually prefer LayerNorm because token representations should be stabilized independently.

Reasons:

```text
1. Batch size can be small.
2. During generation, batch size may be 1.
3. Sequence lengths can vary.
4. Examples in the batch should not affect each other.
5. Autoregressive generation should behave consistently during training and inference.
```

BatchNorm depends on batch statistics. That can be awkward when batch size changes or when generating one sequence at a time.

LayerNorm works the same whether:

```text
B = 1
```

or:

```text
B = 64
```

because each token vector is normalized independently.

---

## Why Normalization Matters

Deep networks can become unstable because activations can grow, shrink, or shift layer by layer.

Without normalization, vectors may become too large:

```text
[100, -80, 300, 50]
```

or too small:

```text
[0.0001, -0.0002, 0.0003]
```

Both make training harder.

LayerNorm keeps token vectors in a controlled numerical range before sending them into attention or feedforward layers.

This helps with:

```text
stable training
better gradient flow
less exploding or shrinking activations
training deeper transformer stacks
```

---

## LayerNorm in a Transformer Block

In modern transformer blocks, LayerNorm is often used in **pre-norm** form:

```python
x = x + self_attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

This means the model normalizes before applying attention or the feedforward MLP.

The structure is:

```text
x
↓
LayerNorm
↓
Self-attention
↓
Residual add
↓
LayerNorm
↓
Feedforward
↓
Residual add
```

Pre-norm is useful because attention and feedforward layers receive cleaner, stabilized inputs.

Street version:

```text
LayerNorm cleans each token vector before attention or MLP processes it.
Residual connections preserve the old information and add the new update.
```

---

## Final Summary

LayerNorm is the normalization method used in transformers because it stabilizes each token independently.

```text
BatchNorm:
normalizes features using batch-level statistics

LayerNorm:
normalizes each token's feature vector independently
```

For transformers:

```text
LayerNorm is preferred because it does not depend on batch size,
does not make examples influence each other,
and works naturally during autoregressive generation.
```

In a transformer block:

```python
x = x + attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

LayerNorm keeps representations numerically stable, while residual connections preserve information and help gradients flow through deep networks.


```python
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
```

# Scaling Up the Transformer Model and Adding Dropout

After building the basic transformer block, the model can be scaled up by defining important architecture, training, and device variables.

Instead of hardcoding values throughout the code, we define variables such as:

```python
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
```

---

## Device Selection

PyTorch can run tensors and models on different hardware backends.

The main options are:

```text
cuda:
NVIDIA GPU acceleration

mps:
Apple Silicon GPU acceleration through Metal

cpu:
regular CPU execution
```

A portable device setup is:

```python
device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(device)
```

This means:

```text
Use CUDA if an NVIDIA GPU is available.
Otherwise use MPS if running on a supported Apple Silicon Mac.
Otherwise fall back to CPU.
```

For an Apple Silicon MacBook, such as an M-series MacBook Pro, the selected device should usually be:

```python
'mps'
```

For an NVIDIA GPU machine, it should usually be:

```python
'cuda'
```

For a machine without GPU acceleration, it will be:

```python
'cpu'
```

---

## Moving the Model and Data to the Device

The model must be moved to the selected device:

```python
model = GPTLanguageModel().to(device)
```

The input and target batches also need to be moved:

```python
xb, yb = get_batch('train')
xb, yb = xb.to(device), yb.to(device)
```

The model and data must be on the same device. Otherwise PyTorch will raise a device mismatch error.

During generation, the starting tensor should also be created on the correct device:

```python
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
```

---

## Main Training Variables

```text
batch_size:
how many sequences are processed in parallel

block_size:
maximum context length, meaning how many previous tokens the model can use

max_iters:
number of training steps

eval_interval:
how often validation loss is checked

learning_rate:
optimizer step size

eval_iters:
number of batches used to estimate validation loss

device:
hardware backend used for training or inference
```

---

## Main Architecture Variables

The most important transformer architecture variables are:

```python
n_embd = 384
n_head = 6
n_layer = 6
```

These mean:

```text
n_embd:
dimension of each token representation

n_head:
number of attention heads in each transformer block

n_layer:
number of transformer blocks stacked
```

If:

```python
n_embd = 384
n_head = 6
```

then each attention head has size:

```python
head_size = n_embd // n_head
```

so:

```text
head_size = 384 // 6 = 64
```

Each head operates in a 64-dimensional communication subspace.

Together:

```text
6 heads × 64 dimensions = 384 dimensions
```

So the model creates multiple attention subspaces while keeping the final token representation size equal to `n_embd`.

---

## Conceptual Meaning

```text
n_embd:
total information space available for each token

n_head:
number of parallel communication channels

head_size:
size of each communication channel

n_layer:
how many times the model repeats communication and computation

device:
where the computation happens
```

Each transformer block lets tokens communicate through multi-head attention and then process that information through a feedforward layer.

Stacking more layers allows the model to build deeper contextual representations.

---

## Dropout

Dropout is a regularization technique. It randomly sets some activations to zero during training.

For example:

```text
[0.2, 0.5, -1.1, 0.7]
```

might become:

```text
[0.2, 0.0, -1.1, 0.0]
```

This prevents the model from relying too heavily on any single neuron, feature, or attention pathway.

In transformers, dropout can be applied to:

```text
attention weights
attention output projections
feedforward layers
residual pathway outputs
```

During training:

```python
model.train()
```

dropout is active.

During evaluation or generation:

```python
model.eval()
```

dropout is turned off.

---

## Why Dropout Helps

As the model gets larger, it has more capacity and can overfit the training data.

Dropout helps prevent this by forcing the model to learn more robust and distributed representations.

Street version:

```text
Scaling up gives the model more brainpower.
Dropout prevents it from memorizing fragile shortcuts.
```

---

## Final Summary

Scaling up the transformer means increasing:

```text
embedding size
number of attention heads
number of transformer layers
context length
batch size
training iterations
```

This gives the model more capacity to learn complex patterns.

Device selection controls where the computation happens:

```text
cuda:
best for NVIDIA GPUs

mps:
best for Apple Silicon Macs

cpu:
fallback option, slower but always available
```

Dropout is added to make the larger model more robust and less likely to overfit.

In short:

```text
n_embd controls the size of each token's information space.
n_head controls how many attention channels exist.
n_layer controls how many times the model refines representations.
device controls where the computation runs.
dropout improves generalization by randomly disabling parts of the network during training.
```


# Stacking Transformer Blocks: How the Information Space Deepens

A transformer block takes the current token representations and applies two main updates:

```python
x = x + self.sa(self.ln1(x))
x = x + self.ffwd(self.ln2(x))
```

Conceptually, this means:

```text
1. Normalize the current representation.
2. Let tokens communicate through multi-head self-attention.
3. Add that communication update back to the original representation.
4. Normalize again.
5. Let each token process its own information through the feedforward MLP.
6. Add that computation update back to the representation.
```

So one transformer block is essentially:

```text
x = x + communication_update
x = x + computation_update
```

---

## What One Transformer Block Does

Before a block, each token has a representation containing:

```text
token identity
position information
previous contextual information
```

The attention part asks:

```text
Which other tokens should this token listen to?
What information should it receive?
```

Multi-head attention creates multiple learned communication subspaces. Each head can route different kinds of information between tokens.

Then the feedforward part asks:

```text
Given the information this token received, how should I transform it locally?
```

The feedforward MLP creates a nonlinear transformation space for each token.

The residual connection makes the update additive:

```text
new representation = old representation + learned update
```

LayerNorm keeps the representation numerically stable before each update.

---

## What Stacking Blocks Does

When we stack transformer blocks:

```python
self.blocks = nn.Sequential(*[
    Block(n_embd, n_head=n_head)
    for _ in range(n_layer)
])
```

the model repeatedly refines the token representations.

For example, if:

```python
n_layer = 6
```

the information flow is:

```text
raw token + position embedding
↓
block 1: basic contextual information
↓
block 2: richer token-token relations
↓
block 3: higher-level phrase/context features
↓
block 4: more abstract semantic/syntactic structure
↓
block 5: deeper prediction-useful features
↓
block 6: final contextual representation
```

These roles are not manually assigned. The model learns useful representations through training.

---

## Information-Space Interpretation

Each block creates another learned transformation of the representation space.

```text
Input space:
token + position embeddings

After attention:
tokens have exchanged information through multiple learned channels

After feedforward:
each token has transformed the received information into new features

After residual:
old information is preserved and the new update is added

After many blocks:
tokens become deeply contextual representations
```

So stacking blocks means:

```text
The model repeatedly builds deeper learned information spaces,
where each token becomes more context-aware after every layer.
```

---

## Final Summary

One transformer block lets tokens:

```text
communicate
compute
update
```

Many transformer blocks let tokens repeatedly:

```text
communicate
compute
refine their meaning
```

This is how raw token embeddings become rich contextual embeddings useful for next-token prediction.

Street version:

```text
One block lets tokens talk, think, and update themselves once.

Many blocks let tokens repeatedly talk, think, and refine their meaning.

That repeated refinement is the core of the transformer architecture.
```

# Roadmap: From Bigram Model to Transformer Language Model

The model starts as a very simple bigram language model and gradually becomes a full transformer.

---

## 1. Bigram Language Model

The bigram model directly learns:

```text
current token → next token
```

It uses a simple lookup table:

```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

Each token directly reads off logits for the next token.

Limitation:

```text
It only knows the current token.
It has no real context.
```

---

## 2. Token Embeddings

Instead of mapping each token directly to next-token logits, we first map tokens into a richer vector space:

```python
tok_emb = self.token_embedding_table(idx)  # (B,T,n_embd)
```

Now each token becomes a learned vector.

Conceptually:

```text
token ID → learned token representation
```

---

## 3. Positional Embeddings

Attention itself has no built-in notion of order, so we add position information:

```python
pos_emb = self.position_embedding_table(torch.arange(T))
x = tok_emb + pos_emb
```

Now each token representation contains:

```text
what the token is
+
where the token is
```

---

## 4. Self-Attention

Self-attention lets tokens communicate.

Instead of each token being isolated, each token can gather information from previous relevant tokens.

Conceptually:

```text
Which previous tokens should I listen to?
What information should I receive from them?
```

This changes the model from:

```text
isolated token representation
```

to:

```text
contextual token representation
```

---

## 5. Multi-Head Attention

One attention head gives one communication pattern.

Multi-head attention gives many communication channels in parallel:

```text
Head 1: one learned relevance pattern
Head 2: another learned relevance pattern
Head 3: another learned relevance pattern
...
```

The heads are concatenated and projected back into the main embedding space.

Conceptually:

```text
multiple learned subspaces for routing contextual information
```

---

## 6. Feedforward Layer

After attention, each token has received contextual information.

The feedforward layer lets each token process that information independently:

```text
attention = tokens communicate
feedforward = each token thinks about what it heard
```

It usually expands and compresses:

```python
n_embd → 4*n_embd → n_embd
```

This gives the model a larger temporary learnable space for transforming features.

---

## 7. Residual Connections

Residual connections preserve old information while adding new updates:

```python
x = x + attention_update
x = x + feedforward_update
```

Conceptually:

```text
new representation = old representation + learned improvement
```

They also help gradients flow backward, making deep transformer stacks trainable.

---

## 8. LayerNorm

LayerNorm stabilizes each token vector before attention or feedforward processing:

```python
x = x + self_attention(layernorm(x))
x = x + feedforward(layernorm(x))
```

It normalizes across the feature dimension of each token.

Conceptually:

```text
clean up each token vector before processing it
```

---

## 9. Dropout

Dropout randomly disables parts of the network during training.

It helps prevent overfitting by forcing the model to learn robust, distributed representations.

Conceptually:

```text
do not rely too heavily on one pathway
```

---

## 10. Stacked Transformer Blocks

A full transformer stacks many blocks:

```python
x = self.blocks(x)
```

Each block does:

```text
communicate
compute
preserve
normalize
```

Stacking blocks repeatedly refines the token representations.

Conceptually:

```text
raw token embedding
↓
basic context
↓
richer token relations
↓
deeper semantic/syntactic features
↓
prediction-useful contextual representation
```

---

## 11. Final Language Model Architecture

The full model becomes:

```text
token IDs
↓
token embedding table
↓
position embedding table
↓
add token + position embeddings
↓
Transformer block 1
↓
Transformer block 2
↓
...
↓
Transformer block n
↓
final LayerNorm
↓
linear layer to vocab size
↓
logits
↓
cross entropy loss / generation
```

In code:

```python
tok_emb = self.token_embedding_table(idx)       # (B,T,n_embd)
pos_emb = self.position_embedding_table(pos)    # (T,n_embd)
x = tok_emb + pos_emb                           # (B,T,n_embd)

x = self.blocks(x)                              # (B,T,n_embd)
x = self.ln_f(x)                                # (B,T,n_embd)
logits = self.lm_head(x)                        # (B,T,vocab_size)
```

---

## Final Intuition

The bigram model predicts from isolated token identity.

The transformer predicts from deep contextual representations.

```text
Bigram:
current token → next token

Transformer:
token + position
↓
repeated communication and computation
↓
contextual representation
↓
next-token logits
```

The transformer builds a deep learned information space where tokens repeatedly communicate, process what they received, and refine their meaning before predicting the next token.

# Information-Space View of `T`, `block_size`, Training, and Generation

A transformer does not train one fixed `block_size × block_size` attention matrix. Instead, it trains shared operators that act on whatever context is currently inside the model.

The important distinction is:

```text
T = number of active tokens currently inside the model

block_size = maximum number of tokens allowed inside the model's context window
```

So the active information space at one forward pass is:

```text
T tokens × n_embd features per token
```

or conceptually:

```text
current contextual workspace = T token representations
```

If `T = 3`, the model has 3 active token representations.

If `T = 8`, the model has 8 active token representations.

The same learned transformer weights operate on both.

---

## Bigram Information Space

A bigram model has a tiny information space:

```text
current token only
```

It always predicts from one token:

```text
P(next | current token)
```

Example:

```text
I have a car with low
```

The bigram model only uses:

```text
low
```

So its usable context space is always:

```text
1 token
```

---

## Transformer Information Space

A transformer has a larger contextual workspace.

If:

```python
block_size = 8
```

then the transformer can use up to 8 tokens of context.

Example:

```text
I have a car with low
```

If this fits inside `block_size`, the transformer can build a contextual information space over:

```text
I, have, a, car, with, low
```

The token `low` can receive information from `car`, `with`, and the rest of the prefix.

So instead of representing only:

```text
low
```

the model can form something like:

```text
low in the context of car
```

This is the main difference from a bigram model.

---

## Attention as Dynamic Information Routing

For the current `T` tokens, attention dynamically creates a `T × T` communication graph.

If:

```text
T = 3
```

the active graph is:

```text
3 tokens communicating with 3 tokens
```

If:

```text
T = 8
```

the active graph is:

```text
8 tokens communicating with 8 tokens
```

But this graph is not stored as a trained parameter.

It is computed fresh from the current token representations using the learned query/key/value projections.

So the learned part is not:

```text
a fixed attention matrix
```

The learned part is:

```text
how to construct attention matrices from token representations
```

The transformer learns the rules of communication, not one fixed communication table.

---

## Training Information Space

During training, suppose:

```python
block_size = 4
```

and we train on:

```text
input:  I, have, a, car
target: have, a, car, with
```

The model predicts all positions in parallel:

```text
I              → have
I have         → a
I have a       → car
I have a car   → with
```

Because of causal masking, each position has a different effective information space:

```text
position 0: context length 1
position 1: context length 2
position 2: context length 3
position 3: context length 4
```

So even though the training chunk has length `block_size`, the model is trained on context lengths:

```text
1, 2, 3, ..., block_size
```

inside every training block.

This is why shorter contexts work during generation.

---

## Generation Information Space

During generation, the active context grows:

```text
I
I have
I have a
I have a car
```

So `T` grows:

```text
T = 1
T = 2
T = 3
T = 4
```

Once the sequence becomes longer than `block_size`, the model cannot keep everything inside the active workspace.

So it keeps only the latest `block_size` tokens.

If:

```python
block_size = 4
```

then:

```text
I have a car with
```

becomes:

```text
have a car with
```

The older token `I` leaves the active information space.

This is a sliding context window.

---

## Information-Space Constraint

`block_size` is the maximum size of the model's contextual workspace.

It limits how many tokens can communicate at once.

So:

```text
Bigram:
active information space = 1 token

Transformer:
active information space = up to block_size tokens
```

The transformer is more powerful because each token can build its representation from many previous tokens, not just the immediately previous one.

But it is still limited:

```text
tokens outside the block_size window are not directly visible
```

---

## Final Conceptual Picture

The transformer maintains a moving contextual information space.

At each forward pass:

```text
1. The current T tokens enter the workspace.
2. Each token has an n_embd-dimensional representation.
3. Multi-head attention builds communication paths among those T tokens.
4. Feedforward layers transform each token's received information.
5. The final token representation is used to predict the next token.
```

During training:

```text
the model learns to use context lengths from 1 up to block_size
```

During generation:

```text
the context grows until block_size, then slides forward
```

So the clean mental model is:

```text
The transformer does not learn one fixed block_size × block_size table.

It learns shared information-processing rules that operate on a variable active context window.

block_size is the maximum size of that window.

T is the current number of tokens inside it.
```